# H&M Dataset — FOAF Decentralized MF Experiment

Mirrors the ML-100K experiment structure exactly:
- Same FOAF gradient-sharing method (user-stratified, random-sampling batch)
- Same graph topologies: Random-2-out, Random-5-out, Scale-Free, Cycle
- Same evaluation metric: RMSE on held-out test set
- Same baselines: CL (centralized), FL (FedAvg), GL (Gossip), LL (local)

**Data source**: Download `articles.csv`, `customers.csv`, `transactions_train.csv`  
from https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations

**Target**: purchase frequency `y_ij = count(customer_i, product_j)` — same as appendix.

## 0. Imports

In [3]:
import pandas as pd
import numpy as np
import datetime
import math
import copy
import os
from dataclasses import dataclass, field
from typing import Dict, Optional
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import networkx as nx
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


## 1. Data Loading & Preprocessing (follows H_M_Data.ipynb)

In [5]:
# ── Load raw files ──────────────────────────────────────────────────────────
DATA_DIR = "."   # change to path containing the three CSVs

df           = pd.read_csv(os.path.join(DATA_DIR, "transactions_train.csv"),
                           dtype={"article_id": str})
customer_df  = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"),
                           dtype={"article_id": str, "product_code": str})
article_df   = pd.read_csv(os.path.join(DATA_DIR, "articles.csv"),
                           dtype={"article_id": str})

df["product_code"] = df["article_id"].map(
    article_df.set_index("article_id")["product_code"])

# ── Week index (0 = most recent week) ───────────────────────────────────────
df["t_dat"] = pd.to_datetime(df["t_dat"])
df["week"]  = (df["t_dat"].max() - df["t_dat"]).dt.days // 7

print("Raw transactions:", len(df))

Raw transactions: 31788324


In [6]:
# ── Filter: top-6 product types, >=120-purchase customers, top-5000 locations
chosen_types = (
    article_df.groupby('product_type_name')['product_code']
    .count().sort_values()[-6:].index
)
chosen_customers = (
    df.groupby('customer_id')['product_code'].nunique()
    .pipe(lambda s: s[s > 120]).index
)
location_customers = customer_df['postal_code'].value_counts()[1:5001].index
customer_df = customer_df[customer_df['postal_code'].isin(location_customers)]

chosen_articles = article_df[article_df['product_type_name'].isin(chosen_types)]
df = df[df['product_code'].isin(chosen_articles['product_code'])]
df = df[df['customer_id'].isin(chosen_customers)]
df = df[df['customer_id'].isin(customer_df['customer_id'])]

df.drop(['t_dat', 'price', 'article_id', 'sales_channel_id'], axis=1, inplace=True)
df['bought'] = 1
df.reset_index(drop=True, inplace=True)

print(f'Filtered transactions: {len(df)} | Unique customers: {df["customer_id"].nunique()}')


Filtered transactions: 216390 | Unique customers: 1765


## 1b. Participant Selection & Train/Test Split

HM preprocessing pipeline:

1. **Deterministic top-K selection**: keep the `TARGET_USERS` most active customers (by total interaction count), after applying an activity floor of `MIN_INTERACTIONS`
2. **Random 75/25 train/test split** on the interaction rows (same as ML-100K `train_test_split(..., test_size=0.25, random_state=42)`)
3. Drop test entries whose user or item is not in train (cold-start removal)
4. Integer-encode users and items (fit on combined set, same as ML-100K)
5. Val split: 80/20 of train for early stopping
6. Save to disk; reload on subsequent runs


In [8]:
# ── Configuration ────────────────────────────────────────────────────────────
TARGET_USERS     = 1000   # keep the top-K most active users
MIN_INTERACTIONS = 20    # activity floor: discard users below this
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"


def select_top_users(df, target_users, min_interactions,
                     user_col='customer_id', item_col='product_code'):
    """
    Deterministic top-K participant selection by total interaction count.
    1. Discard users with fewer than min_interactions total purchases.
    2. Rank remaining users by total interactions (descending).
    3. Keep the top target_users.
    Returns the row-subset of df for selected customers.
    """
    user_counts = (
        df.groupby(user_col)[item_col].count()
        .rename('n_interactions').reset_index()
    )
    eligible = user_counts[
        user_counts['n_interactions'] >= min_interactions
    ].sort_values('n_interactions', ascending=False)

    selected = eligible.head(target_users)[user_col].values
    df_sub = df[df[user_col].isin(selected)].copy()
    df_sub.reset_index(drop=True, inplace=True)

    print(f'  Eligible (>={min_interactions} interactions): {len(eligible)}')
    print(f'  Selected top-{target_users}: {df_sub[user_col].nunique()} users | {len(df_sub)} rows')
    return df_sub


# ── Save-or-load pattern ──────────────────────────────────────────────────────
cache_tag  = f'u{TARGET_USERS}_m{MIN_INTERACTIONS}_top'
train_path = os.path.join(SAMPLED_DATA_DIR, f'train_{cache_tag}.csv')
val_path   = os.path.join(SAMPLED_DATA_DIR, f'val_{cache_tag}.csv')
test_path  = os.path.join(SAMPLED_DATA_DIR, f'test_{cache_tag}.csv')
meta_path  = os.path.join(SAMPLED_DATA_DIR, f'meta_{cache_tag}.csv')

if all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]):
    # ── Fast path: reload ────────────────────────────────────────────────────
    print(f'Loading cached dataset ({cache_tag}) from {SAMPLED_DATA_DIR}/')
    train_df = pd.read_csv(train_path)
    val_df   = pd.read_csv(val_path)
    test_df  = pd.read_csv(test_path)
    meta_df  = pd.read_csv(meta_path)
    N_USERS  = int(meta_df.loc[meta_df['key'] == 'n_users', 'value'].iloc[0])
    N_ITEMS  = int(meta_df.loc[meta_df['key'] == 'n_items', 'value'].iloc[0])
    print(f'  Loaded: {N_USERS} users | {N_ITEMS} items')
    print(f'  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

else:
    # ── Slow path ────────────────────────────────────────────────────────────
    print('Cache not found — running pipeline...')

    # Step 1: deterministic top-K selection on full filtered rows
    df_sampled = select_top_users(
        df, target_users=TARGET_USERS, min_interactions=MIN_INTERACTIONS
    )

    # Step 2: random 75/25 train/test split on interaction rows
    # Mirrors ML-100K: train_test_split(X, y, test_size=0.25, random_state=42)
    X = df_sampled[['customer_id', 'product_code']].values
    y = df_sampled['bought'].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=SEED
    )

    train_df = pd.DataFrame(X_train, columns=['customer_id', 'product_code'])
    test_df  = pd.DataFrame(X_test,  columns=['customer_id', 'product_code'])
    train_df['bought'] = y_train
    test_df['bought']  = y_test

    # Step 3: keep only test rows whose user appears in train
    # (items may be unseen in train — user-only filter, same as ML-100K)
    train_users = set(train_df['customer_id'])
    mask = test_df['customer_id'].isin(train_users)
    n_dropped = (~mask).sum()
    test_df = test_df[mask].copy()
    print(f'  Dropped {n_dropped} test rows with users absent from train')

    # Step 4: integer encoding — fit on combined train+test (mirrors ML-100K)
    all_users = np.unique(
        pd.concat([train_df[['customer_id']], test_df[['customer_id']]])['customer_id']
    )
    all_items = np.unique(
        pd.concat([train_df[['product_code']], test_df[['product_code']]])['product_code']
    )
    customer_id2idx = {c: i for i, c in enumerate(all_users)}
    item_id2idx     = {a: i for i, a in enumerate(all_items)}

    for df_ in [train_df, test_df]:
        df_['customer_id']  = df_['customer_id'].map(customer_id2idx)
        df_['product_code'] = df_['product_code'].map(item_id2idx)

    N_USERS = len(all_users)
    N_ITEMS = len(all_items)

    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    # Step 5: val split — 80/20 of train for early stopping
    n_val    = int(len(train_df) * 0.2)
    val_df   = train_df.tail(n_val).reset_index(drop=True)
    train_df = train_df.head(len(train_df) - n_val).reset_index(drop=True)

    # Step 6: save
    os.makedirs(SAMPLED_DATA_DIR, exist_ok=True)
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path,     index=False)
    test_df.to_csv(test_path,   index=False)
    pd.DataFrame([
        {'key': 'n_users',          'value': N_USERS},
        {'key': 'n_items',          'value': N_ITEMS},
        {'key': 'n_train',          'value': len(train_df)},
        {'key': 'n_test',           'value': len(test_df)},
        {'key': 'target_users',     'value': TARGET_USERS},
        {'key': 'min_interactions', 'value': MIN_INTERACTIONS},
        {'key': 'selection',        'value': 'top_k_deterministic'},
        {'key': 'split',            'value': 'random_75_25'},
    ]).to_csv(meta_path, index=False)

    print(f'  Saved to {SAMPLED_DATA_DIR}/')
    print(f'  Users   : {N_USERS}')
    print(f'  Items   : {N_ITEMS}')
    print(f'  Train   : {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
    print(f'  Density : {len(train_df)/(N_USERS*N_ITEMS)*100:.4f}%')


Cache not found — running pipeline...
  Eligible (>=20 interactions): 1765
  Selected top-1000: 1000 users | 155137 rows
  Dropped 0 test rows with users absent from train
  Saved to /Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled/
  Users   : 1000
  Items   : 11817
  Train   : 93082 | Val: 23270 | Test: 38785
  Density : 0.7877%


## 2. Dataset & DataLoader

In [10]:
class UserRandomSamplingDataset(Dataset):
    """
    Per-user dataset: each __getitem__ returns a random batch of that user's
    interactions.  Mirrors the ML-100K DataLoader design.
    """
    def __init__(self, df: pd.DataFrame, batch_size: int = 10):
        self.batch_size = batch_size
        # Build user -> indices lookup once
        users = torch.tensor(df["customer_id"].values, dtype=torch.long)
        items = torch.tensor(df["product_code"].values, dtype=torch.long)
        targets = torch.tensor(df["bought"].values, dtype=torch.float32)

        unique_users = users.unique()
        self.user_ids = unique_users
        self.user_to_indices = {
            u.item(): (users == u).nonzero(as_tuple=True)[0]
            for u in unique_users
        }
        self.items   = items
        self.targets = targets

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        uid = self.user_ids[idx].item()
        indices = self.user_to_indices[uid]
        bs = min(len(indices), self.batch_size)
        sel = indices[torch.randperm(len(indices))[:bs]]
        # Return (user_idx, item_idx) pairs and targets
        x = torch.stack([
            torch.full((bs,), uid, dtype=torch.long),
            self.items[sel]
        ], dim=1)  # shape: (bs, 2)  col0=user, col1=item
        y = self.targets[sel]  # shape: (bs,)
        return x, y


BATCH_SIZE = 10

train_dataset = UserRandomSamplingDataset(train_df, batch_size=BATCH_SIZE)
val_dataset   = UserRandomSamplingDataset(val_df,   batch_size=BATCH_SIZE)
test_dataset  = UserRandomSamplingDataset(test_df,  batch_size=BATCH_SIZE)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)

print(f"Train loader batches: {len(train_loader)} (= N_USERS = {N_USERS})")

Train loader batches: 1000 (= N_USERS = 1000)
